You potentially need to `pip install pandas` first.

Just run the cell below. Click on the output, not on the code part of the cell; that opens the whole code. If you accidentally open it, go `View -> Collapse Selected Code` to close it again.

In [53]:
group = 'rigi'

import re
from collections import Counter, defaultdict
from itertools import chain, repeat
from pathlib import Path

import pandas as pd
import rich
from rich.table import Table
from rich.text import Text


# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
from IPython.display import display_html, display
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)


########################
# READ ALL ACTIVE JOBS #


# We need to go with length and cut by lenght, because some entries may have spaces, some may be too long, etc.
jobs = !squeue -A {group} -O JobId:20,Name:20,UserName:20,State:20,TimeUsed:20,NumCPUs:20,QOS:20,NumNodes:20,GRES:20,RestartCnt:20,Reason:20
jobs = [[j[i*20:(i+1)*20].strip() for i in range(11)] for j in jobs]
jobs = pd.DataFrame(jobs[1:], columns=jobs[0])



##########################
# READ ALL WORKDIRS EVER #

# _xid_re = re.compile(r'(\d\d)(\d\d)_(\d\d)(\d\d)(\d\d)')
_xid_re = re.compile(r'\d\d\d\d_\d\d\d\d\d\d')
def extract_xid(name):
    if firstmatch := _xid_re.search(name):
        return firstmatch.group()
    return None


basedir = Path('/checkpoint/rigi/bv2/workdirs')
workdirs = [d.name for d in basedir.iterdir() if d.is_dir()]
wd_by_xid = {xid: wd for wd in workdirs if (xid := extract_xid(wd))}

#####################################
# SPLIT INTO CURRENT / RECENT / OLD #
# We do this split to add much more info to recent, and less to old.

NUM_RECENT = 50

hot_runs = {}
cold_runs = {}
for xid, wd in sorted(wd_by_xid.items(), reverse=True):
    states = Counter(jobs[jobs.NAME == xid].STATE)
    if states:
        hot_runs[xid] = {"states": states, "wd": wd}
    else:
        cold_runs[xid] = wd
frozen_runs = {xid: {"wd": cold_runs[xid]} for xid in list(cold_runs)[NUM_RECENT:]}
cold_runs = {xid: {"wd": cold_runs[xid]} for xid in list(cold_runs)[:NUM_RECENT]}

# TODO: Parallelize
for xid, db in chain(zip(hot_runs.keys(), repeat(hot_runs)), zip(cold_runs.keys(), repeat(cold_runs))):
    launchinfo = basedir / db[xid]["wd"] / 'launchinfo.txt'
    if launchinfo.is_file():  # Launched with our sweep launcher
        db[xid]["wus"] = {}
        for wuwd in (basedir / db[xid]["wd"]).iterdir():
            if wuwd.is_dir():
                db[xid]["wus"][wuwd] = (wuwd / "DONE").exists()
        db[xid]["config"] = next(re.finditer(r"bv2/config/(.*?) ", launchinfo.read_text())).group(1)


#########################
# PREPARE VISUALIZATION #


tblH = Table(show_header=True, header_style="bold magenta", show_footer=True, footer_style="bold magenta", box=rich.box.HORIZONTALS)
tblH.add_column("xid", justify="left")
tblH.add_column("usr", justify="left")
tblH.add_column("states", justify="left")
tblH.add_column("wus", justify="right")
tblH.add_column("QoS", justify="left")
tblH.add_column("config", justify="left")

STATE_NAMES = {
    "RUNNING": Text("Run", "green"),
    "PENDING": Text("Pend", "yellow"),
    "REQUEUE_HOLD": Text("Hold", "red"),
    "COMPLETING": Text("End", "blue"),
    # And our own for old jobs:
    True: Text("Done", "green"),
    False: Text("Fail", "red"),
}
# STATE_NAMES = {"RUNNING": "🏃", "PENDING": "⏳", "REQUEUE_HOLD": "🚫"}  # Sadly misaligns columns.

all_states = Counter()
for xid, info in hot_runs.items():
    all_states.update(info["states"])
    xjobs = jobs[jobs.NAME == xid]
    qos = ' '.join(xjobs.QOS.unique().tolist())
    users = ' '.join(xjobs.USER.unique().tolist())
    states = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in info["states"].most_common())
    tblH.add_row(xid, users, states, str(len(info["wus"])), qos, info["config"])

tblH.columns[2].footer = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in all_states.most_common())

tblC = Table(show_header=True, header_style="bold magenta", box=rich.box.HORIZONTALS)
tblC.add_column("xid", justify="left")
tblC.add_column("usr", justify="left")
tblC.add_column("wus", justify="right")
for xid, info in cold_runs.items():
    user = (basedir / info["wd"]).owner()
    states = []
    if nfail := sum(1 for v in info["wus"].values() if v is False):
        states.append(Text(f"{nfail}", "red"))
    if ngood := sum(1 for v in info["wus"].values() if v is True):
        states.append(Text(f"{ngood}", "green"))
    tblC.add_row(xid, user, Text("+").join(states))

rich.print('All currently active runs:', tblH,
           f'Most recent {NUM_RECENT} inactive runs:', tblC,
           f'Very old runs: [{len(frozen_runs)} not shown]')

All currently active runs:
 ─────────────────────────────────────────────────────────────────────────────────────── 
  xid           usr    states                   wus   QoS              config            
 ─────────────────────────────────────────────────────────────────────────────────────── 
  1125_145646   zhai   Run:2                      2   h100_rigi_high   finevision_8n.py  
  1125_141833   zhai   Pend:16                    0   h200_lowest      finevision_8n.py  
  1125_112652   zhai   Run:19 Pend:13            31   h200_lowest      finevision_8n.py  
  1124_215121   zhai   Pend:4                    32   h200_lowest      finevision_8n.py  
  1124_163539   zhai   Pend:15                   15   h200_lowest      finevision_8n.py  
  1124_143658   zhai   Pend:1 Run:1               2   h200_lowest      finevision_8n.py  
  1117_163432   qkv    Hold:2                   120   h100_lowest      code.py           
  1114_221425   qkv    Hold:55                  235   h100_lowest      code.py           
 ─────────────────────────────────────────────────────────────────────────────────────── 
                       Hold:57 Pend:49 Run:22                                            
 ─────────────────────────────────────────────────────────────────────────────────────── 
Most recent 50 inactive runs:
 ────────────────────────── 
  xid           usr    wus  
 ────────────────────────── 
  1126_131428   pplx     1  
  1126_130750   pplx     1  
  1126_125801   pplx     1  
  1126_111614   pplx     1  
  1126_111423   pplx        
  1126_111105   pplx        
  1126_110917   pplx        
  1126_110311   pplx     1  
  1126_101544   pplx     1  
  1126_101511   pplx        
  1126_101222   pplx     1  
  1126_101114   pplx        
  1125_142102   pplx     1  
  1125_141804   pplx        
  1125_134631   pplx     1  
  1125_133724   pplx        
  1125_133611   pplx        
  1125_125825   pplx     4  
  1125_094101   pplx     4  
  1124_212712   pplx    48  
  1124_161824   pplx        
  1124_161749   pplx        
  1124_161701   pplx        
  1124_151118   pplx     1  
  1124_150321   pplx        
  1124_135955   zhai     7  
  1124_123757   zhai     2  
  1124_123418   zhai        
  1120_135847   qkv      1  
  1120_134111   qkv      1  
  1120_133407   qkv      1  
  1120_105504   pplx     2  
  1120_105437   pplx     2  
  1120_102147   pplx        
  1120_102132   pplx        
  1119_152042   pplx     3  
  1119_113236   pplx     4  
  1119_112304   pplx        
  1118_160143   zhai   4+2  
  1118_160130   zhai   1+5  
  1118_160032   zhai     6  
  1118_135300   zhai     3  
  1118_135236   zhai        
  1118_135207   zhai     1  
  1118_131059   zhai   8+1  
  1118_130956   zhai     9  
  1118_130432   zhai   1+8  
  1117_164507   zhai     2  
  1117_164452   zhai     2  
  1117_154745   zhai     2  
 ────────────────────────── 
Very old runs: [202 not shown]

# Quick look at config and logs

### Code setup

In [29]:
from contextlib import contextmanager
from ipywidgets import Output
from IPython.display import display, display_html

# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)

@contextmanager
def show_scrolling(height=200):
    out = Output(layout={"border": "1px solid #ccc", "height": f"{height}px", "overflow": "auto"})
    with out:
        yield
    display(out)

from pathlib import Path
import json
import sws

def print_config(run, height=384):
    workdir = Path('/checkpoint/rigi/bv2/workdirs') / run
    c = sws.Config(**json.loads((workdir / 'config.json').read_text())).finalize()
    with show_scrolling(height):
        print(c)


def print_logs(run, height=256, head=1000, tail=1000, linehead=100, linetail=100):
    def _snip_long_line(line, linehead=linehead, linetail=linetail):
        if len(line) > linehead + linetail:
            return line[:linehead] + " ... <SNIP> ... " + line[-linetail:]
        else:
            return line

    workdir = Path('/checkpoint/rigi/bv2/workdirs') / run
    c = sws.Config(**json.loads((workdir / 'config.json').read_text())).finalize()

    user = workdir.owner()
    full_log = Path(f'/checkpoint/rigi/bv2/slurm_out/{user}/{c.jid}.txt').read_text()
    loglines = full_log.split('\n')
    print(f"First {head} lines:")
    with show_scrolling(height):
        print('\n'.join(map(_snip_long_line, loglines[:head])))
    # import time
    # time.sleep(3)
    print(f"Last {tail} lines:")
    with show_scrolling(height):
        print('\n'.join(map(_snip_long_line, loglines[-tail:])))
    return full_log

### Actual look

In [34]:
print_config('1117_134621/qkv-fv-base_d8_d12-1117_134621-4', height=384)

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

In [33]:
log = print_logs('1117_134621/qkv-fv-base_d8_d12-1117_134621-4', height=192, head=1500, tail=200, linehead=100, linetail=100)

First 1500 lines:


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

Last 200 lines:


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

# Tmp/dev

This is a place to dig deeper or figure out some things. The useful variables are: `jobs` (from `squeue` command), `hot_runs`, and `workdirs` (or sth like `basedir / workdirs[0]`).

In [82]:
jobs.query('NAME == "1114_221425"')

,JOBID,NAME,USER,STATE,TIME,CPUS,QOS,NODES,TRES_PER_NODE,RESTART_COUNT,REASON
0,942235,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,4,Resources
1,942053,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,6,Priority
2,942054,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,6,Priority
3,942240,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
4,942241,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
5,942242,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
6,942243,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
7,942244,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
8,942245,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
9,942246,1114_221425,qkv,PENDING,0:00,192,h100_lowest,1,gres/gpu:8,3,Priority
